In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Spark Session Initialization
# ==============================================================================
print("Initializing Spark Session...")
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Phase 2 Simple SQL Join")
    .getOrCreate()
)
sc = spark.sparkContext

Initializing Spark Session...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/17 13:29:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# 2. Dataset Ingestion
# ==============================================================================

#cLoad Yelp Data 
yelp_file_path = "data/Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json"
print(f"Loading Yelp data from: {yelp_file_path}...")
yelp_df_raw = spark.read.json(yelp_file_path)

print("yelp data count:", yelp_df_raw.count())


# Load Census DP05 (Demographics) Data 
demo_file_path = "data/ACSDP5Y2023.DP05-Data.csv"
print(f"Loading Demographics data from: {demo_file_path}...")

# Read with header=True, which correctly uses the first row
df_demo_raw = spark.read.csv(demo_file_path, header=True, inferSchema=False)
print("Census count :", df_demo_raw.count())

# Filter out the second row, which is the human-readable header.
df_demo_cleaned = df_demo_raw.filter(F.col("NAME") != "Geographic Area Name")


# Load Census DP03 (Economics) Data 
econ_file_path = "data/ACSDP5Y2023.DP03-Data.csv"
print(f"Loading Economics data from: {econ_file_path}...")

# Read with header=True
df_econ_raw = spark.read.csv(econ_file_path, header=True, inferSchema=False)

# Filter out the second row of econ data
df_econ_cleaned = df_econ_raw.filter(F.col("NAME") != "Geographic Area Name")

print("All datasets ingested.")


yelp_df_raw.show(5)

df_demo_raw.show(5)


In [ ]:
# 3. Data Cleaning and Transformation
# ==============================================================================
print("Cleaning and transforming data...")

# --- Clean Yelp Data ---
yelp_df = yelp_df_raw

# --- Clean Demographics (DP05) Data ---
df_demo = (
    df_demo_cleaned.filter(F.col("NAME").startswith("ZCTA5 "))
    .select(
        # This line will go to the name column, and grab the zip code digits becuase the census data is super annoying
        F.substring(F.col("NAME"), 7, 5).alias("zip_code"),
        F.expr("TRY_CAST(regexp_replace(DP05_0001E, ',', '') AS INT)").alias("Total_Population")
    )
)

# --- Clean Economics (DP03) Data ---
df_econ = (
    df_econ_cleaned.filter(F.col("NAME").startswith("ZCTA5 "))
    .select(
        F.substring(F.col("NAME"), 7, 5).alias("zip_code"),
        F.expr("TRY_CAST(regexp_replace(DP03_0063E, ',', '') AS INT)").alias("Median_Income")
    )
)

print("Data cleaning complete.")

In [ ]:
# 4. Exploratory Data Analysis (EDA) with Spark SQL
# ==============================================================================
print("Joining datasets using Spark SQL...")

# Create temporary views to use Spark SQL
yelp_df.createOrReplaceTempView("yelp_view")
df_demo.createOrReplaceTempView("demo_view")
df_econ.createOrReplaceTempView("econ_view")

# SQL query to join the three views
join_query = """
    SELECT
        y.*, 
        d.Total_Population,
        e.Median_Income
    FROM
        yelp_view y
    JOIN
        demo_view d ON y.postal_code = d.zip_code
    JOIN
        econ_view e ON y.postal_code = e.zip_code
"""

# Run the query
df_final_joined = spark.sql(join_query)

In [ ]:
# 4. Exploratory Data Analysis (EDA) with Spark SQL CONT. Mainly Transformations here
# ==============================================================================

#Summary Statistics for Numerical Columns 
print("Summary Statistics for Numerical Columns ---")
numerical_cols = ["stars", "review_count", "Total_Population", "Median_Income"]
df_final_joined.select(numerical_cols).describe().show()

# Count Distinct Values for Key Categorical Columns
print("Distinct Count for Key Categorical Columns ---")
distinct_states = df_final_joined.select("state").distinct().count()
distinct_cities = df_final_joined.select("city").distinct().count()
distinct_postal_codes = df_final_joined.select("postal_code").distinct().count()
print(f"Number of distinct states: {distinct_states}")
print(f"Number of distinct cities: {distinct_cities}")
print(f"Number of distinct postal codes: {distinct_postal_codes}")

# Value Counts for Top Categories
print(" Top 10 States by Business Count ---")
df_final_joined.groupBy("state").count().orderBy(F.col("count").desc()).show(10)

print(" Top 10 Cities by Business Count ---")
df_final_joined.groupBy("city").count().orderBy(F.col("count").desc()).show(10)

print(" Top 10 Postal Codes by Business Count ---")
df_final_joined.groupBy("postal_code").count().orderBy(F.col("count").desc()).show(10)

print(" Top 10 States with Highest Ratings ---")
df_final_joined.groupBy("state") \
    .agg(F.avg("stars").alias("average_rating")) \
    .orderBy(F.col("average_rating").desc()) \
    .show(10)
    
print(" Reviews by Income")
# Define income brackets
income_brackets = F.when(F.col("Median_Income") < 50000, "Low") \
                   .when((F.col("Median_Income") >= 50000) & (F.col("Median_Income") < 85000), "Medium") \
                   .when(F.col("Median_Income") >= 85000, "High") \
                   .otherwise("Unknown")

df_with_brackets = df_final_joined.withColumn("income_bracket", income_brackets)

# Group by the new bracket and get avg stars and avg review count
df_with_brackets.groupBy("income_bracket") \
    .agg(
        F.avg("stars").alias("average_rating"),
        F.avg("review_count").alias("average_reviews"),
        F.count("*").alias("business_count")
    ) \
    .orderBy("average_rating", ascending=False) \
    .show()



print("--- Distribution of Business Ratings ---")
df_final_joined.groupBy("stars") \
    .count() \
    .orderBy(F.col("stars").desc()) \
    .show()
    
print("--- Top 10 Most Reviewed Businesses ---")
df_final_joined.select("name", "city", "state", "review_count", "stars") \
    .orderBy(F.col("review_count").desc()) \
    .show(10)

In [ ]:
# 5. Show Final Result
# ==============================================================================

#print("\n--- Final Joined DataFrame Schema ---")
#df_final_joined.printSchema()

print("\n--- Final Joined Data (Sample) ---")
# This will show all columns, including all from Yelp + the two from census
df_final_joined.show()
print(df_final_joined.count())

#Stop spark session
#spark.stop()

In [ ]:
from pyspark.sql import SparkSession

csv_path = "/Users/wg1911/Desktop/resturant_data.csv" 

df_osm_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

print("--- CSV Data Loaded Successfully ---")
df_osm_raw.printSchema()
df_osm_raw.show(5)

In [ ]:
from pyspark.sql import functions as F

# Assuming 'df_osm_raw' is your loaded OpenStreetMap DataFrame

# --- 1. Define Regex Patterns for Key Tags ---
# The pattern r'\"key\"=>\"(.*?)\"' extracts the value (group 1) following the key.
# This pattern is robust for the common HStore/JSON-like format found in OSM exports.
AMENITY_REGEX = r'\"amenity\"=>\"(.*?)\"'
CUISINE_REGEX = r'\"cuisine\"=>\"(.*?)\"'
DRIVE_THROUGH_REGEX = r'\"drive_through\"=>\"(.*?)\"'

# --- 2. Extract Features and Create a Cleaned DataFrame ---
df_extracted = (
    df_osm_raw
    # 2a. Extract the Amenity Type (e.g., 'fast_food', 'restaurant')
    .withColumn(
        "amenity_type",
        F.regexp_extract(F.col("other_tags"), AMENITY_REGEX, 1)
    )
    # 2b. Extract the Cuisine Type (e.g., 'burger', 'pizza')
    .withColumn(
        "cuisine_type",
        F.regexp_extract(F.col("other_tags"), CUISINE_REGEX, 1)
    )
    # 2c. Create a boolean feature for 'drive_through'
    .withColumn(
        "has_drive_through",
        F.when(
            F.regexp_extract(F.col("other_tags"), DRIVE_THROUGH_REGEX, 1) == "yes",
            F.lit(1)  # Use 1 for True
        ).otherwise(F.lit(0))  # Use 0 for False/Missing
    )
)

# --- 3. Filter for Relevant Competitors/Businesses ---
# Filter to keep only food service businesses you consider competitors.
df_clean_features = df_extracted.filter(
    (F.col("amenity_type") == "fast_food") | 
    (F.col("amenity_type") == "restaurant")
)

# --- 4. Select Final Columns ---
df_final_osm = df_clean_features.select(
    "osm_id", 
    "name", 
    "address", 
    "X", # Corrected from "lon"
    "Y", # Corrected from "lat"
    "amenity_type", 
    "cuisine_type", 
    "has_drive_through"
    # ... and your new census-joined columns later
)

# You may want to rename them for clarity right away:
df_final_osm = df_final_osm.withColumnRenamed("X", "longitude").withColumnRenamed("Y", "latitude")

df_final_osm.show(5, truncate=False)